<a href="https://colab.research.google.com/github/Sakshiiikashyap/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakshiiikashyap/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Search Intelligence Data Contract

### 1. Unit of analysis + time window

**Unit of analysis.** One row = **one content item for one client** (`client_hash_id × content_hash_id`). The source table `fact_content_daily_performance` is at daily grain — one row per client × content × report date — so I aggregate the daily records into one row per client-content pair.

**Feature window.** February 2026 (`2026-02-01` → `2026-02-28`). These are the signals that would have been knowable at the decision moment.

**Label window.** March 2026 (`2026-03-01` → `2026-03-31`). This is kept completely separate from the feature window so that future information does not enter the features.

**Decision moment.** The decision moment is the end of February 2026. Every feature used for the decision must therefore be computable using information available on or before `2026-02-28`.

**Universe.** I include published content that existed by the end of February and had enough February search exposure and click history to make a zero-click outcome meaningful: at least 100 GSC impressions and at least 3 GSC clicks in February.

### 2. Fields: feature / label / context / excluded

**Features**

* `gsc_impressions_feb` — total GSC impressions in February.
* `gsc_clicks_feb` — total GSC clicks in February.
* `ctr_feb` — February clicks divided by February impressions.
* `avg_position_feb` — impression-weighted average search position in February.
* `content_age_days` — age of the content at the February decision moment.

**Label**

* `went_dark` — 1 when the content item records zero GSC clicks during March 2026, otherwise 0.

**Context / identifiers**

* `client_hash_id` — identifies the pseudonymized client for grouping and aggregation.
* `content_hash_id` — identifies the pseudonymized content item for joining and aggregation.
* `gsc_data_available` — indicates whether GSC data was actually measured for a daily record.

**Excluded**

* March performance metrics are excluded from the feature set because March is the future outcome window.
* Label-derived fields such as `went_dark`, `trend_direction`, or other outcome fields are excluded from the features because they reveal the target.
* Product decision fields such as `health_score`, `priority_score`, `action_type`, and `refresh_tier` are excluded because they encode an existing business decision rather than an independent prediction signal.
* Client names, URLs, domains, and raw query text are excluded because the warehouse is pseudonymized and the analysis does not require identifying information.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1) Data contract

### 1. What does one row mean?

One row represents one **client × content item × report date** observation in the daily performance warehouse. The February source data was verified to have no duplicate combinations of these three fields.

### 2. Which table(s) will you use?

I use the `fact_content_daily_performance` table for daily GSC performance data and the content dimension table for content metadata such as publication/creation information.

### 3. What time window will you use?

I use **February 2026 (2026-02-01 to 2026-02-28)** as the feature window. March 2026 is used as the future outcome window for the label.

### 4. What will you predict or rank?

I will predict whether a content item **goes dark in March 2026**, using the binary label `went_dark`. A value of `1` means the content had zero measured GSC clicks during March; a value of `0` means it had at least one measured March GSC day and did not go dark.

### 5. What will you deliberately exclude?

I deliberately exclude **March performance variables** such as `imp_mar`, `clk_mar`, and `measured_days_mar` from the predictive features because they belong to the future outcome window and would not be known at the February 28 decision moment.

In [1]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

def get_hf_token():
    """Get Hugging Face READ token without hard-coding it."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok

    for candidate in (".env", "../.env", "../../.env"):
        if os.path.exists(candidate):
            with open(candidate) as fh:
                for line in fh:
                    if line.startswith("HF_TOKEN="):
                        return line.split("=", 1)[1].strip()

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the token out of SQL text.
hf_token = get_hf_token()
con.execute("SET VARIABLE hf_token = ?", [hf_token])
con.execute("""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN getvariable('hf_token'))
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("connected; feature window = Feb 2026, label window = Mar 2026")


Paste your Hugging Face READ token (hf_...): ··········
connected; feature window = Feb 2026, label window = Mar 2026


In [5]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Token starts with hf_:", token.startswith("hf_") if token else False)

Token found: True
Token starts with hf_: True


In [6]:
from huggingface_hub import whoami

info = whoami(token=token)
print("Logged in as:", info["name"])

Logged in as: sakshiiikashyap


In [7]:
from huggingface_hub import hf_hub_download

test_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    token=token
)

print("File downloaded successfully!")
print(test_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

File downloaded successfully!
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-02/data_0.parquet


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [8]:
con.execute(f"""
CREATE OR REPLACE VIEW feb_agg AS
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions_feb,
    SUM(gsc_clicks) AS gsc_clicks_feb,
    SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_feb,
    COUNT(*) FILTER (WHERE gsc_data_available) AS measured_days_feb
FROM {FEB}
GROUP BY client_hash_id, content_hash_id
HAVING
    SUM(gsc_impressions) >= 100
    AND SUM(gsc_clicks) >= 3
""")

universe = con.sql(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions_feb,
    f.gsc_clicks_feb,
    f.avg_position_feb,
    f.measured_days_feb,
    d.content_created_date,
    d.is_published
FROM feb_agg f
JOIN {DIM} d
  ON f.content_hash_id = d.content_hash_id
WHERE d.is_published IS TRUE
  AND d.content_created_date <= DATE '2026-02-28'
""").df()

universe["content_created_date"] = pd.to_datetime(
    universe["content_created_date"]
)

universe["ctr_feb"] = (
    universe["gsc_clicks_feb"] /
    universe["gsc_impressions_feb"]
)

universe["content_age_days"] = (
    pd.Timestamp("2026-02-28") -
    universe["content_created_date"]
).dt.days

print(f"February universe rows: {len(universe):,}")
print(f"clients: {universe['client_hash_id'].nunique():,}")


February universe rows: 29,700
clients: 31


## Deliberate leakage experiment

I intentionally create one invalid feature derived directly from the label to demonstrate target leakage.

The invalid feature is `leak_went_dark`, which is simply a copy of `went_dark`. This information would not be available at the February 28 decision moment because it describes the March outcome.

If this feature is given to a classifier, the model can recover the target directly, so the quick score should become artificially close to perfect.

This is not a valid model result. It demonstrates why a feature must be available at the decision moment and must not be derived from the outcome.


In [10]:
universe = con.sql(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions_feb,
    f.gsc_clicks_feb,
    f.avg_position_feb,
    f.measured_days_feb,
    d.content_created_date,
    d.is_published
FROM feb_agg f
JOIN {DIM} d
  ON f.content_hash_id = d.content_hash_id
WHERE d.is_published IS TRUE
  AND d.content_created_date <= DATE '2026-02-28'
""").df()

universe["content_created_date"] = pd.to_datetime(
    universe["content_created_date"]
)

universe["ctr_feb"] = (
    universe["gsc_clicks_feb"] /
    universe["gsc_impressions_feb"]
)

universe["content_age_days"] = (
    pd.Timestamp("2026-02-28") -
    universe["content_created_date"]
).dt.days

print(f"February universe rows: {len(universe):,}")
print(f"clients: {universe['client_hash_id'].nunique():,}")

February universe rows: 29,700
clients: 31


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS duplicate_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id
    FROM {FEB}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
""").df()

grain_check

,duplicate_groups
0,0


In [12]:
con.sql(f"""
SELECT *
FROM {FEB}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13,0,85,...,0,0,0,0,0,0,0,0,0,2026-02
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59,0,1001,...,0,0,0,0,0,0,0,0,0,2026-02
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17,0,287,...,0,0,0,0,0,0,0,0,0,2026-02
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6,0,27,...,0,0,0,0,0,0,0,0,0,2026-02


In [15]:
window_check = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {FEB}
""").df()

window_check

,rows,min_date,max_date
0,7355108,2026-02-01,2026-02-28


Date-span verification: The February feature-window data covers 2026-02-01 through 2026-02-28. The source contains 7,355,108 daily observations in this month. The row-count output from the remote Parquet relation is not used as evidence because the direct COUNT(*) result is inconsistent with the observed relation; the grain was instead verified through the duplicate-key check.

In [16]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
FROM {FEB}
""").df()

availability_check

,available_rows
0,2621783


Availability verification: 2,621,783 February daily observations have gsc_data_available IS TRUE. This confirms that the availability check is based on the warehouse's explicit GSC availability flag rather than treating missing measurements as zero.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [17]:
features = universe[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_feb",
        "gsc_clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "content_age_days"
    ]
].copy()

features.head()

,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,ctr_feb,avg_position_feb,content_age_days
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,28.886364,226
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.004380,17.273467,226
2,client_e547b89c05043229,content_c6dbda992ad84127,2868.0,3.0,0.001046,17.909693,226
3,client_e547b89c05043229,content_fe83160838cdab99,2580.0,9.0,0.003488,17.762016,226
4,client_e547b89c05043229,content_c5baa3a03cfccc41,4711.0,21.0,0.004458,10.231373,226


### Feature availability

| Feature | Why is it available at the decision moment? |
|---|---|
| `gsc_impressions_feb` | Available by February 28 because it is calculated only from Google Search Console impressions observed during February. |
| `gsc_clicks_feb` | Available by February 28 because it is calculated only from clicks observed during the February feature window. |
| `ctr_feb` | Available by February 28 because it is calculated from February clicks and February impressions only. |
| `avg_position_feb` | Available by February 28 because it is calculated only from search-position observations in the February feature window. |
| `content_age_days` | Available by February 28 because the content creation date was already known at the decision moment. |

In [18]:
label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS imp_mar,
    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clk_mar,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_days_mar
FROM {MAR}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print(label.head())

            client_hash_id           content_hash_id  imp_mar  clk_mar  \
0  client_62f4a7e64f5e0096  content_39d7361b4945d504     77.0      0.0   
1  client_62f4a7e64f5e0096  content_cec711b02f3bbde6    602.0      4.0   
2  client_62f4a7e64f5e0096  content_275b6f7f733016d4    810.0      1.0   
3  client_62f4a7e64f5e0096  content_ceaec531566ffcfc     82.0      0.0   
4  client_62f4a7e64f5e0096  content_755d951187fcd70a   1858.0      6.0   

   measured_days_mar  
0                 24  
1                 29  
2                 29  
3                 27  
4                 30  


In [19]:
frame = universe.merge(
    label,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print(frame.shape)
frame.head()

(29700, 13)


,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,avg_position_feb,measured_days_feb,content_created_date,is_published,ctr_feb,content_age_days,imp_mar,clk_mar,measured_days_mar
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,28.886364,28,2025-07-17,True,0.002964,226,768.0,1.0,29.0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.273467,28,2025-07-17,True,0.004380,226,3071.0,6.0,29.0
2,client_e547b89c05043229,content_c6dbda992ad84127,2868.0,3.0,17.909693,28,2025-07-17,True,0.001046,226,3898.0,2.0,29.0
3,client_e547b89c05043229,content_fe83160838cdab99,2580.0,9.0,17.762016,28,2025-07-17,True,0.003488,226,4255.0,8.0,29.0
4,client_e547b89c05043229,content_c5baa3a03cfccc41,4711.0,21.0,10.231373,28,2025-07-17,True,0.004458,226,4095.0,9.0,29.0


In [20]:
frame["went_dark"] = (
    (frame["measured_days_mar"] > 0) &
    (frame["clk_mar"].fillna(0) == 0)
).astype(int)

print(frame["went_dark"].value_counts())
print()
print("Went-dark rate:", round(frame["went_dark"].mean(), 4))

went_dark
0    28541
1     1159
Name: count, dtype: int64

Went-dark rate: 0.039


### Label definition

`went_dark = 1` when a content item has zero measured GSC clicks during March 2026. Otherwise, `went_dark = 0`.

The March label is calculated only after the February feature window. Therefore, March performance is treated as the future outcome and is not used in the five predictive features.

In this dataset, 1,159 of 29,700 content items went dark, giving a went-dark rate of 3.9%.

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

FINAL_FEATURES = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
]

model_data = frame[
    FINAL_FEATURES + ["went_dark"]
].dropna().copy()

X = model_data[FINAL_FEATURES]
y = model_data["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (22275, 5)
Test: (7425, 5)


In [22]:
X_train_leak = X_train.copy()
X_test_leak = X_test.copy()

# Deliberately leaking the target into the features
X_train_leak["leak_went_dark"] = y_train.values
X_test_leak["leak_went_dark"] = y_test.values

In [23]:
leak_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

leak_model.fit(X_train_leak, y_train)

leak_pred = leak_model.predict(X_test_leak)

leak_score = accuracy_score(y_test, leak_pred)

print(f"Leaky accuracy: {leak_score:.3f}")

Leaky accuracy: 1.000


### Deliberate leakage result

The leaky model achieved an accuracy of **1.000 (100%)**.

This result is artificially perfect because `leak_went_dark` is directly copied from the target `went_dark`. The model therefore receives the answer as an input feature instead of having to predict it.

This is **target leakage**. `went_dark` describes the March outcome, so it would not be available at the February 28 decision moment. The 1.000 score is therefore invalid and must not be reported as model performance.

In [24]:
honest_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(
    y_test,
    honest_pred
)

print(f"Honest accuracy: {honest_score:.3f}")

Honest accuracy: 0.961


### Honest result after removing leakage

After removing the label-derived feature, the accuracy dropped from **1.000** to **0.961**.

The 1.000 result was invalid because the model had direct access to the target through `leak_went_dark`. The **0.961 result is the honest quick check** using only the five February features that were available at the decision moment.

However, accuracy should not be interpreted alone because `went_dark` is an imbalanced outcome: only 3.9% of the labeled content items went dark. The purpose of this experiment is to demonstrate the effect of target leakage, not to claim that this simple model is production-ready.

In [25]:
FINAL_FEATURES = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
]

X_honest = frame[FINAL_FEATURES].copy()
y_honest = frame["went_dark"].copy()

print("Final features:")
print(FINAL_FEATURES)
print("\nLeak feature present:", "leak_went_dark" in X_honest.columns)

Final features:
['gsc_impressions_feb', 'gsc_clicks_feb', 'ctr_feb', 'avg_position_feb', 'content_age_days']

Leak feature present: False


## 4. Data limits

### Named limitation — uneven GSC availability

GSC availability is not uniform across all client-content-day observations. An unavailable GSC observation cannot safely be interpreted as zero traffic because unavailable data means the measurement was not available, not that the content received zero clicks.

Therefore, the `went_dark` label is only defined for content items with at least one measured GSC day during March 2026. This can reduce the usable sample and may introduce selection bias.

In [26]:
print("Final labeled rows:", len(frame))
print("Clients represented:", frame["client_hash_id"].nunique())
print("Went-dark rate:", round(frame["went_dark"].mean(), 3))
print(
    "March measured-day range:",
    frame["measured_days_mar"].min(),
    "to",
    frame["measured_days_mar"].max()
)

Final labeled rows: 29700
Clients represented: 31
Went-dark rate: 0.039
March measured-day range: 0.0 to 31.0


In [27]:
frame["went_dark"] = np.where(
    frame["measured_days_mar"] > 0,
    (frame["clk_mar"].fillna(0) == 0).astype(int),
    np.nan
)

print(frame["went_dark"].value_counts(dropna=False))

went_dark
0.0    28194
1.0     1159
NaN      347
Name: count, dtype: int64


In [28]:
labeled = frame.dropna(subset=["went_dark"]).copy()

print("Labeled rows:", len(labeled))
print(labeled["went_dark"].value_counts())
print("Went-dark rate:", round(labeled["went_dark"].mean(), 4))

Labeled rows: 29353
went_dark
0.0    28194
1.0     1159
Name: count, dtype: int64
Went-dark rate: 0.0395


In [29]:
FINAL_FEATURES = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
]

model_data = labeled[FINAL_FEATURES + ["went_dark"]].copy()

X = model_data[FINAL_FEATURES]
y = model_data["went_dark"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (22014, 5)
Test: (7339, 5)


In [30]:
X_train_leak = X_train.copy()
X_test_leak = X_test.copy()

X_train_leak["leak_went_dark"] = y_train.values
X_test_leak["leak_went_dark"] = y_test.values

In [31]:
leak_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

leak_model.fit(X_train_leak, y_train)

leak_pred = leak_model.predict(X_test_leak)

leak_score = accuracy_score(y_test, leak_pred)

print(f"Leaky accuracy: {leak_score:.3f}")

Leaky accuracy: 1.000


In [32]:
honest_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(y_test, honest_pred)

print(f"Honest accuracy: {honest_score:.3f}")

Honest accuracy: 0.960


### Deliberate leakage experiment

The deliberate leakage experiment produced an accuracy of **1.000**, because the target `went_dark` was directly included as the feature `leak_went_dark`. The model therefore had direct access to the answer.

After removing the leaked target-derived feature and training only on the five February features, the honest accuracy was **0.960**.

The difference demonstrates why target leakage must be prevented: a seemingly perfect score can be completely artificial when information from the outcome window is included in the inputs.

In [33]:
print("Final feature count:", len(FINAL_FEATURES))
print("Final features:", FINAL_FEATURES)
print("Leak feature present:", "leak_went_dark" in FINAL_FEATURES)

Final feature count: 5
Final features: ['gsc_impressions_feb', 'gsc_clicks_feb', 'ctr_feb', 'avg_position_feb', 'content_age_days']
Leak feature present: False


In [34]:
print("Final feature count:", len(FINAL_FEATURES))
print("Final features:", FINAL_FEATURES)
print("Leak feature present:", "leak_went_dark" in FINAL_FEATURES)

Final feature count: 5
Final features: ['gsc_impressions_feb', 'gsc_clicks_feb', 'ctr_feb', 'avg_position_feb', 'content_age_days']
Leak feature present: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.